# Frozen baseline - ResNet50 V2 on 30-class Genshin

Frozen ImageNet backbone + linear head. 5 seeds `[0,1,2,3,4]`, stratified 80/10/10 split.
Split is seeded separately (`SPLIT_SEED=42`) and is identical across all model seeds
and identical to the fine-tuned notebook's split.

Test set is touched once per seed, using the best-val checkpoint.

In [ ]:
# Colab only. Skip if running locally with dataset_processed_30/ in cwd.
try:
    from google.colab import files
    IN_COLAB = True

except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    !unzip -q dataset_processed_30.zip

In [ ]:
import json
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models

# == Config ==
SEEDS       = [0, 1, 2, 3, 4]
SPLIT_SEED  = 42    # fixed across all seeds; same value in fine-tuned notebook
DATA_ROOT   = "dataset_processed_30"
BATCH_SIZE  = 32
EPOCHS      = 30
LR_HEAD     = 1e-3
TRAIN_FRAC  = 0.8
VAL_FRAC    = 0.1

WEIGHTS     = models.ResNet50_Weights.IMAGENET1K_V2
RESULTS_DIR = "results/frozen"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# == Data ==
# split independent of model seed
transform = WEIGHTS.transforms()
full_ds   = datasets.ImageFolder(DATA_ROOT, transform=transform)
CLASSES   = full_ds.classes
print(f"{len(CLASSES)} classes, {len(full_ds)} images")
print(CLASSES)

class_to_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_ds.samples):
    class_to_indices[label].append(idx)

g_split = torch.Generator().manual_seed(SPLIT_SEED)
train_idx, val_idx, test_idx = [], [], []
for label in sorted(class_to_indices):
    idxs = class_to_indices[label]
    perm = torch.randperm(len(idxs), generator=g_split).tolist()
    shuffled = [idxs[i] for i in perm]
    n = len(shuffled)
    n_test = max(1, int(round((1 - TRAIN_FRAC - VAL_FRAC) * n)))
    n_val  = max(1, int(round(VAL_FRAC * n)))
    test_idx.extend(shuffled[:n_test])
    val_idx.extend(shuffled[n_test:n_test + n_val])
    train_idx.extend(shuffled[n_test + n_val:])

print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
print("split heads:")
print("  train[:5]:", train_idx[:5])
print("  val[:5]:  ", val_idx[:5])
print("  test[:5]: ", test_idx[:5])

train_loader = DataLoader(Subset(full_ds, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(Subset(full_ds, val_idx),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(Subset(full_ds, test_idx),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
def run_frozen(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = models.resnet50(weights=WEIGHTS)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LR_HEAD)

    ckpt_path = f"{RESULTS_DIR}/best_seed{seed}.pth"
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_epoch = 0.0, -1

    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)

        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                v_loss  += criterion(logits, yb).item() * xb.size(0)
                correct += (logits.argmax(1) == yb).sum().item()
                total   += yb.size(0)
        val_loss = v_loss / len(val_loader.dataset)
        val_acc  = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"  epoch {epoch:02d}  train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch
            torch.save({"model_state": model.state_dict(),
                        "epoch": epoch,
                        "best_val_acc": best_val_acc}, ckpt_path)
            print(f"    -> saved (best={best_val_acc:.4f})")

    # == learning curves ==
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["train_loss"], label="train")
    ax1.plot(history["val_loss"],   label="val")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
    ax1.set_title(f"Loss - seed {seed}"); ax1.legend()
    ax2.plot(history["val_acc"])
    ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
    ax2.set_title(f"Val acc - seed {seed}")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/curves_seed{seed}.png", dpi=150)
    plt.close()

    # == reload best, evaluate on test ==
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            preds.extend(model(xb).argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    test_acc = accuracy_score(labels, preds)

    # == row-normalized confusion matrix ==
    cm = confusion_matrix(labels, preds, normalize="true")
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=False, cmap="Blues", vmin=0, vmax=1,
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.title(f"Confusion (row-normalized) - seed {seed}")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/confusion_seed{seed}.png", dpi=150)
    plt.close()

    print(f"  seed {seed}: val={best_val_acc:.4f}  test={test_acc:.4f}  "
          f"(best epoch {best_epoch})")
    return {"seed": seed,
            "best_val": float(best_val_acc),
            "test": float(test_acc),
            "best_epoch": int(best_epoch)}


# == Run all seeds, save incrementally ==
results = []
for seed in SEEDS:
    print(f"\n== seed {seed} ==")
    results.append(run_frozen(seed))
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump({"condition": "frozen",
                   "model": "resnet50",
                   "weights": WEIGHTS.name,
                   "split_seed": SPLIT_SEED,
                   "seeds": SEEDS,
                   "runs": results}, f, indent=2)

vals  = [r["best_val"] for r in results]
tests = [r["test"]      for r in results]

print("\n" + "=" * 60)
print(f"Frozen baseline - ResNet50 ({WEIGHTS.name}), {len(SEEDS)} seeds")
print(f"  val:  {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"  test: {np.mean(tests):.4f} ± {np.std(tests):.4f}")
print("=" * 60)